In [3]:
import pandas as pd

In [5]:
df = pd.read_csv('../backend/logs.csv')

In [6]:
df.head()

,id,colaborador_id,tag,evento,observacao,data_hora
0,1,1.0,TAG001,ENTRADA,Entrada autorizada,2026-05-11 20:05:05.216052
1,2,1.0,TAG001,SAIDA,Saida registrada,2026-05-11 20:05:05.216052
2,3,2.0,TAG002,ENTRADA,Entrada autorizada,2026-05-11 20:05:05.216052
3,4,3.0,TAG003,NEGADO,Colaborador sem permissao,2026-05-11 20:05:05.216052
4,5,4.0,TAG004,NEGADO,Colaborador sem permissao,2026-05-11 20:05:05.216052


In [7]:
df['data_hora'] = pd.to_datetime(df['data_hora'])

In [8]:
df['data'] = df['data_hora'].dt.date

In [9]:
entradas = df[df['evento'] == 'ENTRADA']

entradas_por_dia = entradas.groupby('data').size()

entradas_por_dia

data
2026-05-11    2
2026-05-12    2
dtype: int64

In [10]:
saidas = df[df['evento'] == 'SAIDA']

saidas_por_dia = saidas.groupby('data').size()

saidas_por_dia

data
2026-05-11    1
2026-05-12    2
dtype: int64

In [11]:
invasoes = df[df['evento'] == 'INVASAO']

invasoes_por_dia = invasoes.groupby('data').size()

invasoes_por_dia

data
2026-05-11    2
dtype: int64

In [12]:
negados = df[df['evento'] == 'NEGADO']

negados_por_dia = negados.groupby('data').size()

negados_por_dia

data
2026-05-11    2
dtype: int64

In [13]:
logs_entrada_saida = df[
    df['evento'].isin(['ENTRADA', 'SAIDA'])
].copy()

In [14]:
logs_entrada_saida = logs_entrada_saida.sort_values(
    by=['colaborador_id', 'data_hora']
)

In [15]:
resultado = []

for colaborador_id in logs_entrada_saida['colaborador_id'].unique():

    logs_colaborador = logs_entrada_saida[
        logs_entrada_saida['colaborador_id'] == colaborador_id
    ]

    entrada = None
    tempo_total = pd.Timedelta(0)

    for _, row in logs_colaborador.iterrows():

        if row['evento'] == 'ENTRADA':
            entrada = row['data_hora']

        elif row['evento'] == 'SAIDA' and entrada is not None:

            tempo_total += row['data_hora'] - entrada
            entrada = None

    resultado.append({
        'colaborador_id': colaborador_id,
        'tempo_total': tempo_total
    })

In [16]:
permanencia_df = pd.DataFrame(resultado)

permanencia_df

,colaborador_id,tempo_total
0,1.0,0 days 00:02:00.420409
1,2.0,0 days 00:00:00


In [17]:
permanencia_df['horas'] = (
    permanencia_df['tempo_total']
    .dt.total_seconds() / 3600
)

permanencia_df

,colaborador_id,tempo_total,horas
0,1.0,0 days 00:02:00.420409,0.03345
1,2.0,0 days 00:00:00,0.00000


In [18]:
ranking_invasoes = (
    invasoes
    .groupby('tag')
    .size()
    .sort_values(ascending=False)
)

ranking_invasoes

tag
TAG005    1
TAG006    1
dtype: int64

In [19]:
permanencia_df.to_csv(
    'relatorio_permanencia.csv',
    index=False
)